# **HybridSearch**

Cette recherche hybride combine une recherche par mots-clés et une recherche sémantique avec Reciprocal Rank Fusion (RRF). Les résultats de chaque méthode sont d’abord ramenés au niveau des affaires (`source_id`), puis fusionnés par leur rang. Les scores TF-IDF, BM25 et les distances sémantiques ne sont donc pas comparés directement.

Deux variantes sont évaluées séparément : TF-IDF + sémantique et BM25 + sémantique. Cela permet de comparer équitablement les deux méthodes lexicales sans donner deux voix au lexical dans une fusion à trois listes.

Paramètres : collection `jurisprudence_chunk_500_overlap_240`, `top_k=10` affaires finales, `n_results=700` chunks candidats pour la recherche sémantique et une profondeur lexicale de 700 affaires candidates.

In [56]:
import json
import sys
from math import log2
from pathlib import Path
from statistics import mean

import chromadb
from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer

TOP_K = 10 
N_RESULTS = 100
RRF_K = 60
COLLECTION_NAME = "jurisprudence_chunk_300_overlap_140"

backend_candidates = [Path.cwd(), Path.cwd() / "Backend", Path.cwd().parent]
backend_candidates.extend(parent / "Backend" for parent in Path.cwd().parents)
BACKEND_DIR = next(
    (
        candidate.resolve()
        for candidate in backend_candidates
        if (candidate / "Retreival" / "KeywordSearch.py").is_file()
        and (candidate / "Collections").is_dir()
    ),
    None,
)
if BACKEND_DIR is None:
    raise FileNotFoundError("Could not locate the Backend directory")
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from Retreival.KeywordSearch import KeywordSearch_BM25, KeywordSearch_TFIDF, tokenizer
from Retreival.SemanticSearch import semanticSearch

print(f"Backend: {BACKEND_DIR}")
print(f"Collection: {COLLECTION_NAME}; top_k={TOP_K}; n_results={N_RESULTS}; RRF k={RRF_K}")

Backend: C:\Users\aliai\Desktop\M2 I2A\RAG Agentique\Backend
Collection: jurisprudence_chunk_300_overlap_140; top_k=10; n_results=100; RRF k=60


## **Connecting to the collection**

In [57]:
client = chromadb.PersistentClient(path=str(BACKEND_DIR / "Collections"))
collection = client.get_collection(name=COLLECTION_NAME)
print(f"Collection contains {collection.count()} chunks")

Collection contains 38976 chunks


## **Loading chunks for keyword search**

Chroma is read in batches so every chunk can be indexed for TF-IDF and BM25. Semantic retrieval continues to use the collection's existing vector index.

In [58]:
def get_all_documents(collection, batch_size=500):
    documents = []
    ids = []
    metadatas = []
    offset = 0

    while True:
        batch = collection.get(
            limit=batch_size,
            offset=offset,
            include=["documents", "metadatas"],
        )
        batch_ids = batch["ids"]
        batch_documents = batch["documents"] or []
        batch_metadatas = batch["metadatas"] or []

        if not batch_ids:
            break
        if not (len(batch_ids) == len(batch_documents) == len(batch_metadatas)):
            raise ValueError("Chroma returned misaligned documents, IDs, and metadata")

        ids.extend(batch_ids)
        documents.extend(batch_documents)
        metadatas.extend(batch_metadatas)
        offset += len(batch_ids)

    if not documents:
        raise ValueError("The Chroma collection contains no documents")

    return {"documents": documents, "ids": ids, "metadatas": metadatas}


keyword_index = get_all_documents(collection)
keyword_index["tfidf_vectorizer"] = TfidfVectorizer()
keyword_index["tfidf_vectors"] = keyword_index["tfidf_vectorizer"].fit_transform(
    keyword_index["documents"]
)
keyword_index["bm25_tokens"] = tokenizer(
    keyword_index["documents"],
    add_special_tokens=False,
    truncation=False,
)["input_ids"]
keyword_index["bm25_index"] = BM25Okapi(keyword_index["bm25_tokens"], k1=1.25)

print(f"Keyword indexes built for {len(keyword_index['documents'])} chunks")

Keyword indexes built for 38976 chunks


## **Reciprocal Rank Fusion**

RRF gives each affair a score of `sum(1 / (RRF_K + rank))` across the ranked lists. Only rank positions are used, so TF-IDF/BM25 scores and Chroma distances remain on their own scales. Each input list is already deduplicated by `source_id`.

In [59]:
def reciprocal_rank_fusion(rankings, top_k=TOP_K, rrf_k=RRF_K):
    if top_k <= 0 or rrf_k < 0:
        raise ValueError("top_k must be positive and rrf_k must be non-negative")

    fused = {}
    for method_name, results in rankings.items():
        seen_source_ids = set()
        for rank, result in enumerate(results, start=1):
            source_id = result.get("source_id") or result.get("metadatas", {}).get("source_id")
            if not source_id:
                raise ValueError(f"{method_name} result at rank {rank} has no source_id")
            if source_id in seen_source_ids:
                raise ValueError(f"{method_name} returned duplicate source_id {source_id}")
            seen_source_ids.add(source_id)

            if source_id not in fused:
                fused[source_id] = {
                    "source_id": source_id,
                    "rrf_score": 0.0,
                    "ranks": {},
                    "document": result.get("document", ""),
                    "id": result.get("id"),
                    "metadatas": result.get("metadatas", {}),
                    "best_rank": rank,
                }

            case = fused[source_id]
            case["rrf_score"] += 1 / (rrf_k + rank)
            case["ranks"][method_name] = rank
            if rank < case["best_rank"]:
                case["document"] = result.get("document", "")
                case["id"] = result.get("id")
                case["metadatas"] = result.get("metadatas", {})
                case["best_rank"] = rank

    ranked_cases = sorted(
        fused.values(),
        key=lambda case: (-case["rrf_score"], case["source_id"]),
    )
    for case in ranked_cases:
        case.pop("best_rank")
    return ranked_cases[:top_k]


def hybridSearch(query, keyword_search, top_k=TOP_K, n_results=N_RESULTS, rrf_k=RRF_K):
    if n_results <= 0:
        raise ValueError("n_results must be positive")

    keyword_results = keyword_search(
        query=query,
        top_k=n_results,
        index=keyword_index,
    )
    semantic_results = semanticSearch(
        collection=collection,
        index=keyword_index,
        query=query,
        n_results=n_results,
        top_k=n_results,
    )

    return reciprocal_rank_fusion(
        {"keyword": keyword_results, "semantic": semantic_results},
        top_k=top_k,
        rrf_k=rrf_k,
    )

## **Searching for cases**

Each hybrid variant fuses one keyword ranking with the semantic ranking. The example uses a French query about termination of an employment contract.

In [60]:
query = "Je cherche des affaires liées à la résiliation d’un contrat de travail."

hybrid_tfidf_results = hybridSearch(
    query=query,
    keyword_search=KeywordSearch_TFIDF,
)
hybrid_bm25_results = hybridSearch(
    query=query,
    keyword_search=KeywordSearch_BM25,
)

print("TF-IDF + semantic (RRF)")
for rank, result in enumerate(hybrid_tfidf_results, start=1):
    print(
        rank,
        result["source_id"],
        round(result["rrf_score"], 6),
        result["ranks"],
        result["metadatas"].get("themes", ""),
    )

print("\\nBM25 + semantic (RRF)")
for rank, result in enumerate(hybrid_bm25_results, start=1):
    print(
        rank,
        result["source_id"],
        round(result["rrf_score"], 6),
        result["ranks"],
        result["metadatas"].get("themes", ""),
    )

TF-IDF + semantic (RRF)
1 6253c844bd3db21cbdd84be2 0.031281 {'keyword': 2, 'semantic': 6} agent commercial, contrat, fin
2 658096f03ea7c8c1120de567 0.026541 {'keyword': 23, 'semantic': 9} 
3 658096f23ea7c8c1120de93c 0.025709 {'keyword': 22, 'semantic': 14} 
4 6253c84cbd3db21cbdd84d1f 0.024955 {'keyword': 6, 'semantic': 42} conventions internationales, accords et conventions divers, convention de bruxelles du 27 septembre 1968
5 6253c839bd3db21cbdd84a77 0.024884 {'keyword': 48, 'semantic': 4} conventions internationales, accords et conventions divers, convention de bruxelles du 27 septembre 1968
6 658096e63ea7c8c1120dd5bf 0.02481 {'keyword': 30, 'semantic': 13} 
7 658096e63ea7c8c1120dd54c 0.024476 {'keyword': 50, 'semantic': 5} 
8 6253c845bd3db21cbdd84c29 0.024187 {'keyword': 41, 'semantic': 10} bail commercial, prix, fixation du loyer du bail renouvelé
9 658096ee3ea7c8c1120de35e 0.023088 {'keyword': 17, 'semantic': 39} 
10 65b94aa35a029d9e20db0392 0.022972 {'keyword': 92, 'semantic': 1

## **Evaluation**

The evaluation uses the same query/reference file as the keyword search notebook and scores distinct cases at `top_k=10`. Relevance labels are provisional, so the scores should be treated as indicative until manually validated.

In [61]:
evaluation_path = BACKEND_DIR / "Retreival" / "keyword_evaluation.json"
with evaluation_path.open(encoding="utf-8") as file:
    evaluation_data = json.load(file)
queries = evaluation_data["queries"]


def evaluate_hybrid(keyword_search, method_name, evaluation_queries):
    per_query = []
    for query_data in evaluation_queries:
        relevant_ids = set(query_data["relevant_source_ids"])
        if not relevant_ids:
            raise ValueError(f"{query_data['query_id']} has no relevance labels")

        ranked_results = hybridSearch(
            query=query_data["query"],
            keyword_search=keyword_search,
            top_k=TOP_K,
            n_results=N_RESULTS,
            rrf_k=RRF_K,
        )
        ranked_ids = [result["source_id"] for result in ranked_results]
        if len(ranked_ids) != len(set(ranked_ids)):
            raise ValueError(f"{method_name} returned duplicate cases")

        row = {
            "method": method_name,
            "query_id": query_data["query_id"],
            "query": query_data["query"],
            "retrieved_source_ids": ranked_ids,
        }
        found = sum(source_id in relevant_ids for source_id in ranked_ids[:TOP_K])
        row[f"precision@{TOP_K}"] = found / TOP_K
        row[f"recall@{TOP_K}"] = found / len(relevant_ids)

        dcg = sum(
            1 / log2(rank + 1)
            for rank, source_id in enumerate(ranked_ids[:TOP_K], start=1)
            if source_id in relevant_ids
        )
        ideal_count = min(len(relevant_ids), TOP_K)
        ideal_dcg = sum(
            1 / log2(rank + 1)
            for rank in range(1, ideal_count + 1)
        )
        row[f"ndcg@{TOP_K}"] = dcg / ideal_dcg if ideal_dcg else 0.0
        row["mrr@10"] = next(
            (1 / rank for rank, source_id in enumerate(ranked_ids[:10], start=1)
             if source_id in relevant_ids),
            0.0,
        )
        per_query.append(row)

    metric_names = [f"precision@{TOP_K}", f"recall@{TOP_K}", f"ndcg@{TOP_K}", "mrr@10"]
    summary = {
        "method": method_name,
        "queries": len(per_query),
        **{metric: mean(row[metric] for row in per_query) for metric in metric_names},
    }
    return summary, per_query


hybrid_tfidf_summary, hybrid_tfidf_evaluation = evaluate_hybrid(
    KeywordSearch_TFIDF,
    "TF-IDF + semantic (RRF)",
    queries,
)
hybrid_bm25_summary, hybrid_bm25_evaluation = evaluate_hybrid(
    KeywordSearch_BM25,
    "BM25 + semantic (RRF)",
    queries,
)

print(hybrid_tfidf_summary)
print(hybrid_bm25_summary)

{'method': 'TF-IDF + semantic (RRF)', 'queries': 15, 'precision@10': 0.04, 'recall@10': 0.22777777777777777, 'ndcg@10': 0.1495641449995505, 'mrr@10': 0.14333333333333334}
{'method': 'BM25 + semantic (RRF)', 'queries': 15, 'precision@10': 0.04, 'recall@10': 0.22777777777777777, 'ndcg@10': 0.1689837132939576, 'mrr@10': 0.1895238095238095}
